#Ingesting and tansforming JSON files - Hands-on

##Pre-Check: Query to inspect JSON file in text format

In [0]:
%sql
select * from text.`/Volumes/workspace/default/learning/json_dataset/json_data.json` limit 10;
-- Note that the JSON file is in lines format

##Pre-Check: Query calling read_files() to read the JSON file (as STRING data type)

In [0]:
%sql
select * 
from read_files(
    '/Volumes/workspace/default/learning/json_dataset/json_data.json',
    format => 'json'
) limit 10;
-- OBS! Output in tabular format
-- OBS! Notice the columns 'key' and 'value' are strings and encoded as base64. The column 'timestamp' is in UNIX format

##CTAS for the bronze layer

In [0]:
%sql
-- Create de Delta Table
create table if not exists bronze_table as 
select * 
from read_files(
    '/Volumes/workspace/default/learning/json_dataset/json_data.json',
    format => 'json'
);

-- Display the table
select * from bronze_table;


###Query to decode the columns key and value

In [0]:
%sql
-- Step1: Decode base64 strings as binary
select
    key as encoded_key,
    unbase64(key) as decoded_key,
    value as encoded_value,
    unbase64(value) as decoded_value
from bronze_table; 
-- OBS! Note that the decoded columns are Binary type. Still not readable

###Apply transformations to encoded columns

In [0]:
%sql
-- Step2: Create a table where those binaries become readable strings
create or replace table bronze_table_decoded as 
select
    cast(unbase64(key) as string) as decoded_key,
    offset,
    partition,
    timestamp,
    cast(unbase64(value) as string) as decoded_value -- decoded_value is now a JSON-formatted string column
from bronze_table; 

select * from bronze_table_decoded;
-- OBS! Note that decoded_value is a JSON formatted string column. Next we will see how to work with it

###Query to flattening the 'decoded_value' column (JSON-formatted string)
When the JSON string has a "simple" structure, you can use this approach (col_name:key) to access the values

In [0]:
%sql
-- How to extract a column from a column containing a JSON formatted string
select
    decoded_value,
    decoded_value:event_type as event_type, -- the value of event_type is a string
    decoded_value:payload as payload,-- the value of payload is another JSON formatted string
    decoded_value:items as array_of_json_strings -- to list the values of the nested-array of JSON formatted string, but in this case, it´s easier to work with STRUCT
from bronze_table_decoded;

###Query to flattening the JSON formatted string via STRUCT data type conversion
Valid approach for complex JSON structures

In [0]:
%sql
-- Query to get one row from the decoded table
SELECT decoded_value FROM bronze_table_decoded LIMIT 1;

####schema_of_json() function

In [0]:
%sql
-- Query to derive the schema from a JSON formatted string row
SELECT schema_of_json(decoded_value) AS schema
FROM
  (SELECT decoded_value FROM bronze_table_decoded LIMIT 1);

####from_json() function

In [0]:
%sql
-----------------------------------------------------------------------------------
-- The goal is to convert the JSON formatted string column to a JSON STRUCT column
-----------------------------------------------------------------------------------

-- Step 1: Get the STRUCT type of the JSON formatted string. Derive the schema from the current data.
-- Copy one of the JSON formatted string rows to the clipboard to determine the derived schema via the schema_of_json function
select schema_of_json('{"event_type":"customer_created","payload":{"id":1,"profile":{"name":"Alice","age":30,"tags":["premium","beta_user"]},"history":["created","welcome_msg"]}}') as schema;
-- The output is a string with the struct of the JSON row: "STRUCT<event_type: STRING, payload: STRUCT<history: ARRAY<STRING>, id: BIGINT, profile: STRUCT<age: BIGINT, name: STRING, tags: ARRAY<STRING>>>>"

-- Step 2: Apply the schema to the JSON formatted string column
-- Copy and paste the schema output into the from_json function
select
  from_json(
    decoded_value,
    'STRUCT<event_type: STRING, payload: STRUCT<history: ARRAY<STRING>, id: BIGINT, profile: STRUCT<age: BIGINT, name: STRING, tags: ARRAY<STRING>>>>'
  ) as struct_column
from bronze_table_decoded;
-- The output is in nice STRUCT format. More readable than the JSON formatted string



####Create a new table with the data transformation

In [0]:
%sql
-- Step 3: Create a new bronze table with the STRUCT column
create or replace table bronze_table_struct
as
select * except (decoded_value),
  from_json(
    decoded_value,
    'STRUCT<event_type: STRING, payload: STRUCT<history: ARRAY<STRING>, id: BIGINT, profile: STRUCT<age: BIGINT, name: STRING, tags: ARRAY<STRING>>>>'
  ) as struct_column
  from bronze_table_decoded;

In [0]:
%sql
-- Step 4: Query the table
select * from bronze_table_struct limit 1;
-- OBS! Note that struct_colum is STRUCT data type. Next we will see how to work with it

##Extract fields, nested fields, nested-arrays from the STRUCT column

In [0]:
%sql
-- Easily access the different fields of the STRUCT column in the select statement
select
  decoded_key,
  struct_column.event_type as event_type, -- Field
  struct_column.payload.history as json_history, -- Nested-field from payload. It is an array	
  struct_column.payload.history[0] as first_element, -- Nested-field from payload
  array_size(struct_column.payload.history) as number_of_elements
from bronze_table_struct limit 1;

In [0]:
%sql
SELECT decoded_key, struct_column.*, struct_column.payload.* FROM bronze_table_struct limit 1;

##Explode array

In [0]:
%sql
-- Explode arrays
-- It will transform each element of the array column into a separate row
create or replace table bronze_explode_array as
select
  decoded_key,
  struct_column.payload.history as json_history,
  array_size(struct_column.payload.history) as number_of_elements,
  explode(struct_column.payload.history) as items_in_array
from bronze_table_struct;

select * from bronze_explode_array;


## Parse JSON formatted String as a VARIANT value

In [0]:
%sql
create or replace table bronze_variant as 
select
  decoded_key,
  offset,
  partition,
  timestamp,
  parse_json(decoded_value) as json_variant_value
from bronze_table_decoded;

select * from bronze_variant;

In [0]:
%sql
-- Parse the VARIANT data type column using : to create the desired table
select
  json_variant_value,
  json_variant_value:event_type :: STRING as event_type, -- Obtain the value of event_type and cast to a string
  json_variant_value:payload:history :: ARRAY<STRING> as history
from bronze_variant;